# GNN-PCNA — Full Retrain from Scratch (seed=42, deterministic)

This notebook reproduces the `pcna_reproduced/best.ckpt` checkpoint with the fixed seed wiring.

**Before running:** `Runtime → Change runtime type → T4 GPU`  
**Expected total time:** ~2–3 hours on T4  
**Output:** New checkpoint committed back to the repo, validation numbers updated.

### Steps
1. Install deps  
2. Clone repo  
3. Pretrain on CryptoSite (42 structures, seed=42)  
4. Fine-tune on 8GLA PCNA  
5. Run validation metrics  
6. Save checkpoint + results to Google Drive  

In [ ]:
# Cell 1: Verify GPU and install PyTorch Geometric
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Switch to GPU runtime: Runtime → Change runtime type → T4 GPU'

# Install PyG (must match torch version)
TORCH_VER = torch.__version__.split('+')[0]  # e.g. '2.1.0'
CUDA_VER  = 'cu121'  # Colab default as of 2025
print(f'Installing PyG for torch={TORCH_VER}, cuda={CUDA_VER}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    f'torch-scatter',
    f'torch-sparse',
    'torch-geometric',
    '--find-links', f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html'
], check=True)

# Other deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'biopython==1.83', 'scikit-learn==1.5.2', 'tqdm'
], check=True)

import torch_geometric
print('PyG:', torch_geometric.__version__, '— ready')

In [ ]:
# Cell 2: Clone repo
import os, subprocess

REPO = 'https://github.com/Reshwant-Borra/GNN_PCNA.git'
if not os.path.exists('/content/GNN_PCNA'):
    subprocess.run(['git', 'clone', REPO, '/content/GNN_PCNA'], check=True)
else:
    subprocess.run(['git', '-C', '/content/GNN_PCNA', 'pull'], check=True)

os.chdir('/content/GNN_PCNA')
print('Working dir:', os.getcwd())
print('Graphs available:', len(list(__import__('pathlib').Path('data/graphs_xl').glob('*.pt'))))

In [ ]:
# Cell 3: Build train/val split directories from cryptosite_split.json
import json, shutil
from pathlib import Path

split = json.loads(Path('data/splits/cryptosite_split.json').read_text())
print(f'Split — train:{len(split["splits"]["train"])} val:{len(split["splits"]["val"])} test:{len(split["splits"]["test"])}')
print(f'seed={split["seed"]}  val_frac={split["val_frac"]}  test_frac={split["test_frac"]}')

SRC = Path('data/graphs_xl')
for split_name in ('train', 'val', 'test'):
    dst = Path(f'data/graphs_xl_{split_name}_split')
    dst.mkdir(exist_ok=True)
    # clear stale links
    for f in dst.glob('*.pt'):
        f.unlink()
    for pdb_id in split['splits'][split_name]:
        src_file = SRC / f'{pdb_id}.pt'
        if src_file.exists():
            shutil.copy2(src_file, dst / f'{pdb_id}.pt')

train_count = len(list(Path('data/graphs_xl_train_split').glob('*.pt')))
val_count   = len(list(Path('data/graphs_xl_val_split').glob('*.pt')))
print(f'Copied: {train_count} train graphs, {val_count} val graphs')

In [ ]:
# Cell 4: PRETRAIN on CryptoSite (seed=42, deterministic)
# Expected: ~1.5 hours on T4, best around epoch 8-12
import subprocess, sys

cmd = [
    sys.executable, '-m', 'src.training.train',
    '--model_size',   'xl',
    '--node_dim',     '520',
    '--phase',        'pretrain',
    '--train_dir',    'data/graphs_xl_train_split',
    '--val_dir',      'data/graphs_xl_val_split',
    '--epochs',       '50',
    '--batch_size',   '4',
    '--lr',           '3e-4',
    '--weight_decay', '1e-4',
    '--seed',         '42',
    '--ckpt_dir',     'checkpoints/retrain_pretrain',
]
print('Command:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False)
print('Return code:', result.returncode)
assert result.returncode == 0, 'Pretrain failed'

In [ ]:
# Cell 5: Check pretrain output
import json
from pathlib import Path

meta_files = list(Path('checkpoints/retrain_pretrain').glob('*meta*.json'))
if meta_files:
    meta = json.loads(meta_files[0].read_text())
    print('Pretrain result:')
    for k, v in meta.items():
        print(f'  {k}: {v}')
else:
    print('No meta file — listing dir:')
    for f in Path('checkpoints/retrain_pretrain').iterdir():
        print(' ', f.name)

In [ ]:
# Cell 6: FINE-TUNE on 8GLA PCNA (seed=42, deterministic)
# Expected: ~30-45 minutes on T4
import subprocess, sys
from pathlib import Path

# Find the pretrain checkpoint
pretrain_ckpts = list(Path('checkpoints/retrain_pretrain').glob('best*.ckpt'))
assert pretrain_ckpts, 'No pretrain checkpoint found'
pretrain_ckpt = str(pretrain_ckpts[0])
print('Using pretrain checkpoint:', pretrain_ckpt)

cmd = [
    sys.executable, 'scripts/finetune_pcna.py',
    '--pretrain',   pretrain_ckpt,
    '--model_size', 'xl',
    '--epochs',     '100',
    '--lr',         '3e-4',
    '--wd',         '1e-4',
    '--seed',       '42',
    '--ckpt_dir',   'checkpoints/retrain_finetune',
]
print('Command:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False)
print('Return code:', result.returncode)
assert result.returncode == 0, 'Fine-tune failed'

In [ ]:
# Cell 7: Run AOH gate check (sanity — must PASS >= 0.7)
import subprocess, sys
from pathlib import Path

ckpt_files = list(Path('checkpoints/retrain_finetune').glob('best*.ckpt'))
assert ckpt_files, 'No finetune checkpoint found'
new_ckpt = str(ckpt_files[0])
print('Testing checkpoint:', new_ckpt)

result = subprocess.run([
    sys.executable, 'scripts/aoh_gate_check.py',
    '--ckpt', new_ckpt,
    '--model', 'xl',
], capture_output=False)
assert result.returncode == 0, 'AOH gate check failed'

In [ ]:
# Cell 8: Run held-out validation metrics
import subprocess, sys
from pathlib import Path

new_ckpt = str(list(Path('checkpoints/retrain_finetune').glob('best*.ckpt'))[0])

result = subprocess.run([
    sys.executable, 'scripts/compute_validation_metrics.py',
    '--ckpt', new_ckpt,
], capture_output=False)

# Print summary
import json
ext = json.loads(Path('data/results/extended_metrics.json').read_text())
print('\n=== RETRAINED MODEL RESULTS ===')
p = ext['pooled']
print(f'Pooled AUROC:  {p["auroc"]:.4f}  CI [{p["auroc_ci_95"][0]:.4f}, {p["auroc_ci_95"][1]:.4f}]')
print(f'Pooled AUPRC:  {p["auprc"]:.4f}  (lift {p["lift_above_trivial"]:.1f}x)')
print(f'Pooled MCC:    {p["mcc"]:.4f}  (threshold={p["mcc_threshold"]:.2f})')
print(f'EF1%:          {p["ef1"]:.2f}x')
cv = ext.get('clean_val_mean', ext['val_mean'])
ct = ext.get('clean_test_mean', ext['test_mean'])
print(f'Clean val AUROC:  {cv["auroc"]:.4f}')
print(f'Clean test AUROC: {ct["auroc"]:.4f}')

In [ ]:
# Cell 9: Overwrite canonical checkpoints + save to Google Drive
import shutil, os
from pathlib import Path
from google.colab import drive

# Paths
new_pretrain = list(Path('checkpoints/retrain_pretrain').glob('best*.ckpt'))[0]
new_finetune = list(Path('checkpoints/retrain_finetune').glob('best*.ckpt'))[0]
new_pretrain_meta = list(Path('checkpoints/retrain_pretrain').glob('*meta*.json'))
new_finetune_meta = list(Path('checkpoints/retrain_finetune').glob('*meta*.json'))

# Overwrite canonical checkpoint locations
shutil.copy2(new_pretrain, 'checkpoints/reproduced_xl/best.ckpt')
shutil.copy2(new_finetune, 'checkpoints/pcna_reproduced/best.ckpt')
if new_pretrain_meta:
    shutil.copy2(new_pretrain_meta[0], 'checkpoints/reproduced_xl/best_meta.json')
if new_finetune_meta:
    shutil.copy2(new_finetune_meta[0], 'checkpoints/pcna_reproduced/best_meta.json')
print('Canonical checkpoints updated')

# Save to Google Drive
drive.mount('/content/drive')
outdir = Path('/content/drive/MyDrive/GNN_PCNA_retrain')
outdir.mkdir(exist_ok=True)

for src in [new_pretrain, new_finetune,
            Path('data/results/extended_metrics.json'),
            Path('data/results/homology_check.json')]:
    dst = outdir / src.name
    shutil.copy2(src, dst)
    print(f'Saved {src.name} → Drive ({src.stat().st_size/1e6:.1f} MB)')

In [ ]:
# Cell 10: Commit new checkpoint + metrics back to GitHub
# You need a GitHub token with repo write access.
# Set it in Colab Secrets (key icon in left sidebar) as GH_TOKEN.
import os, subprocess
from google.colab import userdata

try:
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None

if not token:
    print('No GH_TOKEN secret found.')
    print('To push manually after downloading from Drive:')
    print('  cp <drive>/best_pcna_reproduced.ckpt checkpoints/pcna_reproduced/best.ckpt')
    print('  cp <drive>/best_pretrain.ckpt checkpoints/reproduced_xl/best.ckpt')
    print('  git add checkpoints/ data/results/extended_metrics.json')
    print('  git commit -m "retrain: seed=42 deterministic checkpoint"')
    print('  git push')
else:
    repo_url = f'https://{token}@github.com/Reshwant-Borra/GNN_PCNA.git'
    subprocess.run(['git', 'config', 'user.email', 'advay.awesomer@gmail.com'], check=True)
    subprocess.run(['git', 'config', 'user.name', 'Advay'], check=True)
    subprocess.run(['git', 'add',
        'checkpoints/reproduced_xl/best.ckpt',
        'checkpoints/reproduced_xl/best_meta.json',
        'checkpoints/pcna_reproduced/best.ckpt',
        'checkpoints/pcna_reproduced/best_meta.json',
        'data/results/extended_metrics.json',
    ], check=True)
    subprocess.run(['git', 'commit', '-m',
        'retrain: seed=42 deterministic checkpoint\n\nFull retrain from scratch with fixed seed_everything()\ncudnn.deterministic=True, benchmark=False.\nCheckpoint now bit-reproducible from committed code.'
    ], check=True)
    subprocess.run(['git', 'remote', 'set-url', 'origin', repo_url], check=True)
    subprocess.run(['git', 'push', 'origin', 'main'], check=True)
    print('Pushed to GitHub')

## After Colab

If GH_TOKEN was set and push succeeded, pull locally:
```bash
git pull
python scripts/aoh_gate_check.py --ckpt checkpoints/pcna_reproduced/best.ckpt --model xl
```

If no token, download from Drive and copy manually:
```bash
cp ~/Downloads/best_finetune.ckpt checkpoints/pcna_reproduced/best.ckpt
cp ~/Downloads/best_pretrain.ckpt checkpoints/reproduced_xl/best.ckpt
git add checkpoints/ data/results/
git commit -m 'retrain: seed=42 deterministic checkpoint'
git push
```

**Expected results (should be close but not identical to original):**
- AOH gate: PASS (mean >= 0.7)
- Clean test AUROC: 0.90–0.95
- Clean val AUROC: 0.60–0.70

If AOH gate FAILS, the fine-tuning diverged — rerun Cell 6 with `--lr 1e-4`.